# BART + LoRA Training Run

Official fine-tuning run for the milestone. Trains on a fixed 30k-row subset
of the training data (out of ~317k available) -- a deliberate time/quality
tradeoff to keep training to roughly 30-45 min on a Colab T4, rather than
the ~7 hours a full-dataset run would take. See `docs/model_comparison.md`
for the tradeoff discussion.

**If running in a fresh Colab tab**, run this notebook's Cell 1 first --
it mounts Drive, clones/pulls the repo, installs dependencies, and restores
processed data from your Drive backup if available. If nothing is found
locally or in a Drive backup, it will tell you to run
`00_colab_setup.ipynb`'s full download+preprocess flow first.

For hyperparameter exploration, alternate subset sizes, or other ad hoc
experiments, use a notebook under `experiments/` instead -- keep this
notebook clean and reproducible.

In [ ]:
# 1. Full environment setup for this tab: mount Drive, clone/pull repo, install deps,
# load API keys + git identity, confirm GPU, and restore processed data (locally or
# from Drive backup) -- everything 00_colab_setup.ipynb does, condensed for this notebook.
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_DIR = '/content/Review-Summarization-LLM'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    %cd /content
    !git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
    %cd {REPO_DIR}

!pip install -q -r requirements.txt

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
github_token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "jrsheffie@gmail.com"
!git config --global user.name "jrsheffie"

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'

if not os.path.exists('data/processed/train.csv'):
    if os.path.exists(f'{BACKUP_DIR}/train.csv'):
        print('train.csv not found locally -- restoring from Drive backup...')
        !mkdir -p data/processed data/raw
        !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
        !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null
        print('Restored from Drive backup.')
    else:
        raise AssertionError(
            "Processed data not found locally or in Drive backup -- "
            "run 00_colab_setup.ipynb's full download+preprocess flow (cells 6-9) first."
        )
else:
    print('Processed data found locally.')

print('Environment ready.')

In [ ]:
# 2. Create a fixed, reproducible 30k-row training subset (+ proportional val subset)
import pandas as pd

TRAIN_SUBSET_SIZE = 30000
VAL_SUBSET_SIZE = 3000
SEED = 42

train_df = pd.read_csv('data/processed/train.csv', engine='python', on_bad_lines='warn')
val_df = pd.read_csv('data/processed/val.csv', engine='python', on_bad_lines='warn')

train_subset = train_df.sample(n=min(TRAIN_SUBSET_SIZE, len(train_df)), random_state=SEED)
val_subset = val_df.sample(n=min(VAL_SUBSET_SIZE, len(val_df)), random_state=SEED)

train_subset.to_csv('data/processed/train_subset_30k.csv', index=False)
val_subset.to_csv('data/processed/val_subset_30k.csv', index=False)

print(f'Train subset: {len(train_subset)} rows')
print(f'Val subset: {len(val_subset)} rows')

In [ ]:
# 3. Run training on the subset (~30-45 min on a T4)
import time
from training.train_bart import train

start = time.time()
train(
    train_path='data/processed/train_subset_30k.csv',
    val_path='data/processed/val_subset_30k.csv',
    output_dir='outputs/bart_lora_30k',
)
elapsed = time.time() - start
print(f'Training completed in {elapsed/60:.1f} minutes')

In [ ]:
# 4. Back up the trained adapter to Drive immediately (checkpoints are gitignored --
# too large for GitHub -- so Drive is the only persistent copy across sessions)
BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
!mkdir -p {BACKUP_DIR}/bart_lora_30k
!cp -r outputs/bart_lora_30k/final/* {BACKUP_DIR}/bart_lora_30k/
print('Checkpoint backed up to Drive.')

## Sanity check: generate on a few held-out validation examples

Quick qualitative check that fine-tuning actually improved on the zero-shot
BART baseline seen in `00_colab_setup.ipynb`'s smoke test -- not a
substitute for the full ROUGE/BERTScore evaluation (see Next Steps below).

In [ ]:
# 5. Load the fine-tuned adapter and compare its output to the reference Summary
from transformers import BartTokenizer, BartForConditionalGeneration
from peft import PeftModel
import pandas as pd

base_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
tokenizer = BartTokenizer.from_pretrained('outputs/bart_lora_30k/final')
model = PeftModel.from_pretrained(base_model, 'outputs/bart_lora_30k/final')
model.eval()

sample = pd.read_csv('data/processed/val_subset_30k.csv', engine='python', on_bad_lines='warn').sample(5, random_state=1)

for _, row in sample.iterrows():
    inputs = tokenizer(row['Text'], return_tensors='pt', max_length=1024, truncation=True)
    output_ids = model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True)
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print('Review:', row['Text'][:150], '...')
    print('Reference Summary:', row['Summary'])
    print('Fine-tuned BART Summary:', generated)
    print('---')

## Next steps

Run the full reference-based evaluation on this checkpoint's predictions
across the val/test set using `evaluation/evaluate.py reference` mode
(ROUGE-1/2/L + BERTScore against the real `Summary` column), then update
`docs/evaluation_report.md` with the results.